## Phase I Project Proposal
### Market Efficiency in Prediction Markets

#### Name: Rushil Damania, DS 3000



### Introduction

Prediction markets like Kalshi and Polymarket allow users to trade on the outcomes of real-world events, from elections to sports outcomes. These markets are efficient in theory, meaning prices should reflect all available information and converge across platforms. However, market inefficiencies can show up on these prediction markets due to liquidity constraints, delayed information, or platform-specific user bases. This leads me to two  questions: 1) Do systematic price discrepancies exist across different market conditions (high vs. low volume, different event types) that could signal trading opportunities? 2) How quickly do prediction market prices adjust to new information, and does adjustment speed vary by market characteristics like liquidity or event type?

Practical applications: identifying inefficiencies could inform trading strategies, understanding price adjustment speeds reveals how well these markets aggregate information and how effecient they really are.

### Data Collection

I plan to use Kalshi's REST API to collect data on active prediction markets across multiple event categories. While the current sample focuses primarily on NFL player performance and game outcome markets due to platform availability at this time, Kalshi offers markets spanning sports, politics, economics, and other events. By collecting data on currently active markets, I can analyze pricing patterns and market efficiency across different event types and trading conditions. Kalshi's public API is straightforward to use and I demonstrate below how I can read in the relevant data (even if it is not completely clean):


In [5]:
# Get the API and Load the Credentials to Access it
import requests
import pandas as pd

# Kalshi PUBLIC API base URL
base_url = "https://api.elections.kalshi.com/trade-api/v2"

markets_endpoint = f"{base_url}/markets"
params = {
    'limit': 100,
    'status': 'open'
}

response = requests.get(markets_endpoint, params=params)
markets_data = response.json()

# Setting up empty lists
tickers = []
titles = []
prices = []
volumes = []
statuses = []

# Get first 40 markets regardless of volume
for market in markets_data.get('markets', [])[:40]:  # Just take first 40
    tickers.append(market.get('ticker', 'N/A'))
    titles.append(market.get('title', 'N/A'))
    prices.append(market.get('last_price', 0) / 100)
    volumes.append(market.get('volume', 0))
    statuses.append(market.get('status', 'N/A'))

# Create DataFrame
df_markets = pd.DataFrame({
    'ticker': tickers,
    'title': titles,
    'last_price': prices,
    'volume': volumes,
    'status': statuses
})

df_markets.head()

,ticker,title,last_price,volume,status
0,KXMVENFLMULTIGAMEEXTENDED-2025850AC7CA713-CCE3...,"yes Xavier Worthy,yes Isiah Pacheco",0.0,0,active
1,KXMVENFLMULTIGAMEEXTENDED-2025E27117F9EF0-8B18...,"yes Travis Etienne Jr.,yes Xavier Worthy,yes O...",0.0,0,active
2,KXMVENFLMULTIGAMEEXTENDED-2025E27117F9EF0-D0DE...,"yes Travis Etienne Jr.,yes Xavier Worthy,yes O...",0.0,0,active
3,KXMVENFLMULTIGAMEEXTENDED-202594DAEF55196-F56F...,"yes Kansas City,yes Over 48.5 points scored",0.0,0,active
4,KXMVENFLMULTIGAMEEXTENDED-2025E27117F9EF0-7D2B...,"yes Travis Etienne Jr.,yes Xavier Worthy,yes O...",0.0,0,active


### Data Usage and Remaining Issues



The above data set is mostly clean already, but there are still some issues to address. The main challenge is that the market titles represent multi-player parlays (e.g., "yes Travis Etienne Jr., yes Xavier Worthy, yes O..."), which will need to be parsed to extract individual player names and market types for more granular analysis. Many markets also currently show zero volume and zero price, indicating they haven't begun trading yet. I'll need to filter these out or collect data closer to game time when markets become more active going forward.
While we haven't covered ML models in class yet, both of my questions could be addressed through supervised learning techniques. For examining price discrepancies and arbitrage opportunities, I could use classification to predict whether a market represents a "good" or "poor" trading opportunity based on features like spread and volume. For understanding how markets adjust to new information, regression models could predict the magnitude of price movements following specific types of events.